In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import torch
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.transforms import v2 as T
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes
import gradio as gr

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class names + colors
class_names = ["background", "pen", "plug", "shoe"]

int_colors = [
    (0,0,0),        # background
    (0,255,0),      # pen (green)
    (255,0,0),      # plug (red)
    (0,0,255)       # shoe (blue)
]

num_classes = len(class_names)

# Load trained model
checkpoint_path = "/content/drive/MyDrive/DL_Project/2025-12-03_02-54-13/maskrcnn_resnet50_fpn_v2.pth"

model = maskrcnn_resnet50_fpn_v2(weights=None, num_classes=num_classes)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully!")

# Preprocessing transforms
infer_tfms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])


Model loaded successfully!


In [9]:
from torchvision.ops import nms

def clean_detections_per_class(
    boxes, labels, scores, masks,
    score_thresh=0.2,     # higher = fewer boxes
    nms_iou=0.2,          # lower = more aggressive merging
    max_per_class=3       # set to 1 if you only want ONE box per class
):
    """
    Apply score threshold + class-wise NMS + (optional) top-K per class.
    Returns filtered boxes, labels, scores, masks.
    """
    # 1) filter by score
    keep = scores > score_thresh
    boxes  = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]
    masks  = masks[keep]

    if len(scores) == 0:
        return boxes, labels, scores, masks

    # 2) class-wise NMS
    final_keep = []

    for c in labels.unique():
        cls_idx = torch.nonzero(labels == c, as_tuple=False).flatten()
        cls_boxes  = boxes[cls_idx]
        cls_scores = scores[cls_idx]

        # NMS on this class
        kept_local = nms(cls_boxes, cls_scores, nms_iou)

        # optional: keep only top-K per class
        if max_per_class is not None and len(kept_local) > max_per_class:
            topk = cls_scores[kept_local].argsort(descending=True)[:max_per_class]
            kept_local = kept_local[topk]

        final_keep.append(cls_idx[kept_local])

    final_keep = torch.cat(final_keep)

    return (
        boxes[final_keep],
        labels[final_keep],
        scores[final_keep],
        masks[final_keep],
    )

from torchvision.transforms import v2 as T
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes

def segment_image(input_pil, score_thresh=0.2, mask_thresh=0.3):
    # 0) original
    orig = input_pil.convert("RGB")

    # 1) preprocess
    img_tensor = infer_tfms(orig)[None].to(device)

    # 2) forward pass
    with torch.no_grad():
        output = model(img_tensor)[0]

    # 3) move to CPU
    boxes  = output["boxes"].cpu()
    labels = output["labels"].cpu()
    scores = output["scores"].cpu()
    masks  = output["masks"].cpu()   # (N,1,H,W)

    # 4) convert masks to (N,H,W) before cleaning
    masks = masks.squeeze(1)

    # 5) clean detections: strong score + NMS + (optionally) top-K per class
    boxes, labels, scores, masks = clean_detections_per_class(
        boxes, labels, scores, masks,
        score_thresh=score_thresh,
        nms_iou=0.3,       # aggressive, reduces clustered boxes
        max_per_class=3    # set to 1 if you want ONLY ONE BOX per class
    )

    if len(scores) == 0:
        # nothing left after filtering
        return orig, orig, orig

    # 6) binarize masks
    masks = (masks > mask_thresh)

    # 7) colors & labels
    colors = [int_colors[int(lbl)] for lbl in labels]
    label_names = [
        f"{class_names[int(lbl)]} ({s*100:.1f}%)"
        for lbl, s in zip(labels, scores)
    ]

    # 8) base image tensor
    img_uint8 = T.ToImage()(orig)

    # Panel 2: masks + boxes
    annotated_all = draw_segmentation_masks(
        image=img_uint8.clone(),
        masks=masks,
        alpha=0.4,
        colors=colors,
    )
    annotated_all = draw_bounding_boxes(
        image=annotated_all,
        boxes=boxes,
        labels=label_names,
        colors=colors,
        width=6,
        font_size=24,
    )
    annotated_all_pil = T.ToPILImage()(annotated_all)

    # Panel 3: boxes only
    boxes_only = draw_bounding_boxes(
        image=img_uint8.clone(),
        boxes=boxes,
        labels=label_names,
        colors=colors,
        width=6,
        font_size=24,
    )
    boxes_only_pil = T.ToPILImage()(boxes_only)

    return orig, annotated_all_pil, boxes_only_pil

In [14]:
demo = gr.Interface(
    fn=segment_image,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=[
        gr.Image(type="pil", label="Original Image"),
        gr.Image(type="pil", label="All Detections (Masks + Boxes)"),
        gr.Image(type="pil", label="All Detections (Boxes With Highest confidence Only)"),
    ],
    title="Mask R-CNN Instance Segmentation — Pen / Plug / Shoe",
    description=(
        "Upload an image. The app will show: (1) the original image, "
        "(2) all detected objects with masks + boxes, and "
        "(3) all detected objects with bounding boxes only."
    ),
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://07cf9cacbf7fcb888b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://07cf9cacbf7fcb888b.gradio.live
